In [1]:
from rag.loader import resolve_pdf_path

pdf_path = resolve_pdf_path()
print(pdf_path)

/home/aiuser/rag_mini_pjt/data/ai_basic_law/인공지능 발전과 신뢰 기반 조성 등에 관한 기본법(법률)(제20676호)(20260122).pdf


In [2]:
from rag.loader import load_documents

documents = load_documents()

print("전체 청크 수:", len(documents))

for document in documents[:3]:
    print("=" * 100)
    print(document.metadata)
    print(document.page_content[:1000])

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


전체 청크 수: 47
{'source': '/home/aiuser/rag_mini_pjt/data/ai_basic_law/인공지능 발전과 신뢰 기반 조성 등에 관한 기본법(법률)(제20676호)(20260122).pdf', 'chunk_id': 8, 'page': 3, 'pages': '3', 'heading': '제1장 총칙', 'caption': '', 'num_tokens': 173, 'parser': 'docling', 'chunker': 'HybridChunker', 'articles': '제1조(목적) | 제2조(정의)'}
제1장 총칙
제1조(목적) 이 법은 인공지능의 건전한 발전과 신뢰 기반 조성에 필요한 기본적인 사항을 규정함으로써 국민의 권익과 존엄성을 보호하고 국민의 삶의 질 향상과 국가경쟁력을 강화하는 데 이바지함을 목적으로 한다.
제2조(정의) 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2026. 1. 20.>
{'source': '/home/aiuser/rag_mini_pjt/data/ai_basic_law/인공지능 발전과 신뢰 기반 조성 등에 관한 기본법(법률)(제20676호)(20260122).pdf', 'chunk_id': 9, 'page': 3, 'pages': '3', 'heading': '제1장 총칙', 'caption': '', 'num_tokens': 655, 'parser': 'docling', 'chunker': 'HybridChunker', 'articles': '제2조(정의)'}
[관련 조문: 제2조(정의)]
제1장 총칙
1. "인공지능"이란 학습, 추론, 지각, 판단, 언어의 이해 등 인간이 가진 지적 능력을 전자적 방법으로 구현한 것을 말 한다.
2. "인공지능시스템"이란 다양한 수준의 자율성과 적응성을 가지고 주어진 목표를 위하여 실제 및 가상환경에 영향 을 미치는 예측, 추천, 결정 등의 결과물을 추론하는 인공지능 기반 시스템을 말한다.
3. "인공지능기술"이란 인공지능을 구현하기 위하여 필요한 하

In [3]:
from rag.pipeline import build_index
from rag.vectorstore import COLLECTION_NAME

print("컬렉션 이름:", COLLECTION_NAME)

vector_store = build_index(
    recreate=True,
)

print("Qdrant 인덱싱 완료")

컬렉션 이름: ai_basic_law_project


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Qdrant 인덱싱 완료


In [4]:
from common.qdrant import get_qdrant_client
from rag.vectorstore import COLLECTION_NAME

client = get_qdrant_client()

result = client.count(
    collection_name=COLLECTION_NAME,
    exact=True,
)

print("저장된 청크 수:", result.count)

저장된 청크 수: 47


In [5]:
from rag.vectorstore import get_vector_store
from rag.retriever import create_mmr_retriever

vector_store = get_vector_store()

retriever = create_mmr_retriever(
    vector_store=vector_store,
    k=4,
    fetch_k=12,
    lambda_mult=0.5,
)

print("MMR Retriever 준비 완료")

MMR Retriever 준비 완료


In [6]:
question = "고영향 인공지능 사업자가 준수해야 하는 의무는 무엇인가?"

retrieved_documents = retriever.invoke(question)

for rank, document in enumerate(retrieved_documents, start=1):
    metadata = document.metadata

    print("=" * 100)
    print(f"검색 순위: {rank}")
    print(f"청크 ID: {metadata.get('chunk_id')}")
    print(f"PDF 페이지: {metadata.get('pages')}")
    print(f"제목: {metadata.get('heading')}")
    print(f"토큰 수: {metadata.get('num_tokens')}")
    print("-" * 100)
    print(document.page_content)

검색 순위: 1
청크 ID: 46
PDF 페이지: 14
제목: 제4장 인공지능윤리 및 신뢰성 확보
토큰 수: 384
----------------------------------------------------------------------------------------------------
제4장 인공지능윤리 및 신뢰성 확보
제34조(고영향 인공지능과 관련한 사업자의 책무) ① 인공지능사업자는 고영향 인공지능 또는 이를 이용한 제품ㆍ서비 스를 제공하는 경우 고영향 인공지능의 안전성ㆍ신뢰성을 확보하기 위하여 다음 각 호의 내용을 포함하는 조치를 대통령령으로 정하는 바에 따라 이행하여야 한다.
1. 위험관리방안의 수립ㆍ운영
2. 기술적으로 가능한 범위에서의 인공지능이 도출한 최종결과, 인공지능의 최종결과 도출에 활용된 주요 기준, 인 공지능의 개발ㆍ활용에 사용된 학습용데이터의 개요 등에 대한 설명 방안의 수립ㆍ시행
3. 이용자 보호 방안의 수립ㆍ운영
4. 고영향 인공지능에 대한 사람의 관리ㆍ감독
5. 안전성ㆍ신뢰성 확보를 위한 조치의 내용을 확인할 수 있는 문서의 작성과 보관
검색 순위: 2
청크 ID: 19
PDF 페이지: 6
제목: 제2장 인공지능의 건전한 발전과 신뢰 기반 조성을 위한 추진체계
토큰 수: 528
----------------------------------------------------------------------------------------------------
[관련 조문: 제10조(분과위원회 등)]
제2장 인공지능의 건전한 발전과 신뢰 기반 조성을 위한 추진체계
- ② 위원회는 인공지능등 관련 특정 현안을 논의하기 위하여 필요한 경우 특별위원회를 둘 수 있다.
- ③ 위원회는 인공지능등 관련 사항을 전문적으로 검토하기 위하여 관계 전문가 등으로 구성된 자문단을 둘 수 있다.
- ④ 위원회는 정부 내 인공지능 주요 시책의 수립과 사업의 효율적인 추진을 위하여 대통령령으로 정하는 인공지능 책임관으로 구성되는 인공지능책임관협의회

In [13]:
# 최종 답변 생성
from rag.pipeline import answer_question

result = answer_question(
    question,
    retriever=retriever,
)

print(result["answer"])

고영향 인공지능 사업자는 인공지능 또는 이를 이용한 제품ㆍ서비스를 제공하는 경우, 그 인공지능이 고영향 인공지능에 해당하는지 사전에 검토해야 합니다. [S1]

필요한 경우 과학기술정보통신부장관에게 고영향 인공지능에 해당하는지 여부의 확인을 요청할 수 있습니다. [S1]

또한 인공지능사업자는 제4장 제1항 각 호에 따른 사항, 즉 인공지능 수명주기 전반에 걸친 위험의 식별ㆍ평가 및 완화와 인공지능 관련 안전사고를 모니터링하고 대응하는 위험관리체계를 구축해야 하며, 그 이행 결과를 과학기술정보통신부장관에게 제출해야 합니다. [S1]

문서에서 확인할 수 없는 사항은 고영향 인공지능 사업자의 추가적인 구체적 의무입니다. [S1]


In [14]:
for rank, document in enumerate(result["sources"], start=1):
    print("=" * 100)
    print(f"[S{rank}] PDF {document.metadata.get('pages')}페이지")
    print(document.page_content)

[S1] PDF 14페이지
제4장 인공지능윤리 및 신뢰성 확보
1. 인공지능 수명주기 전반에 걸친 위험의 식별ㆍ평가 및 완화
2. 인공지능 관련 안전사고를 모니터링하고 대응하는 위험관리체계 구축
3. ② 인공지능사업자는 제1항 각 호에 따른 사항의 이행 결과를 과학기술정보통신부장관에게 제출하여야 한다.
4. ③ 과학기술정보통신부장관은 제1항 각 호에 따른 사항의 구체적인 이행 방식 및 제2항에 따른 결과 제출 등에 필 요한 사항을 정하여 고시하여야 한다.
5. 제33조(고영향 인공지능의 확인) ① 인공지능사업자는 인공지능 또는 이를 이용한 제품ㆍ서비스를 제공하는 경우 그 인공지능이 고영향 인공지능에 해당하는지에 대하여 사전에 검토하여야 하며, 필요한 경우 과학기술정보통신부장관 에게 고영향 인공지능에 해당하는지 여부의 확인을 요청할 수 있다.
6. ② 과학기술정보통신부장관은 제1항에 따른 요청이 있는 경우 고영향 인공지능 해당 여부를 확인하여야 하며, 필요
7. 한 경우 전문위원회를 설치하여 관련 자문을 받을 수 있다.
8. ③ 과학기술정보통신부장관은 고영향 인공지능의 기준과 예시 등에 관한 가이드라인을 수립하여 보급할 수 있다.
9. ④ 그 밖에 제1항에 따른 확인 절차 등에 관하여 필요한 사항은 대통령령으로 정한다.
[S2] PDF 11페이지
제2절 인공지능기술 개발 및 인공지능산업 활성화
6. 소장의 책임과 권한에 관한 규정
2. ⑤ 인공지능연구소는 업종별ㆍ기능별로 다음 각 호의 사업을 수행한다.
1. 인공지능기술 연구개발
2. 인공지능기술과 다른 기술 및 학제 간 융합에 관한 연구개발
3. 인공지능기술 연구개발 성과 관리ㆍ이전ㆍ활용 및 사업화
4. 인공지능기술 연구개발 전문인력 양성
5. 인공지능기술 연구개발 관련 국제교류ㆍ국제협력
6. 그 밖에 인공지능기술 연구개발에 필요한 사항
9. ⑥ 정부와 지방자치단체는 예산의 범위에서 인공지능연구소의 운영과 사업 추진 등에 필요한 경비를 지원할 수 있 다.
10. ⑦ 인공지능연구소는 운영에 필요한 재원을 조성하기